# Register a whole mouse brain streamed from S3 and resample it out of core

This notebook registers two whole mouse brains published as OME-Zarr in a
public S3 bucket, and resamples one onto the grid of the other at full
resolution without ever downloading a full-resolution volume:

1. `from_ngff_zarr` opens both stores anonymously over S3; the arrays are lazy.
2. [ITKElastix](https://github.com/InsightSoftwareConsortium/ITKElastix)
   registers the coarsest level of the pyramids, which is small enough to fetch
   in a few seconds. The transform lives in physical space, so it applies at
   any level.
3. `itk_transform_resample` applies the transform at a fine level. For every
   output block it asks `itk_transform_resample_bounding_box` which region of
   the moving image is needed, reads only the S3 chunks inside it, and
   resamples through `itkwasm-downsample`. The blocks are tasks of one Dask
   graph that reference the moving chunks directly, so a chunk that several
   blocks need crosses the network once.
4. `to_ngff_zarr` writes the lazy result to a local OME-Zarr block by block, so
   neither the input nor the output has to fit in memory.

The last section benchmarks the resample level by level, from local disk and
from S3.

The data are expansion-assisted selective plane illumination microscopy
(ExaSPIM) volumes from the Allen Institute for Neural Dynamics, in the
`aind-open-data` bucket. Each brain is a four-level pyramid: level 0 is
1776 × 1331 × 513 voxels at 30 × 30 × 40 µm (1.2 billion voxels, 2.4 GB of
`uint16`), level 3 is 2.4 million voxels. The two brains were chosen because
they share an acquisition orientation; other specimens in the cohort are
mirrored along X or Z and would need a flip before a rigid registration can
succeed.

Requirements beyond ngff-zarr: `pip install "ngff-zarr[remote]" itk-elastix matplotlib`.
The 2D companion example that builds the block-wise resample by hand is
[ITKElastix example 23](https://github.com/InsightSoftwareConsortium/ITKElastix/blob/main/examples/ITK_Example23_NgffZarrMultiscaleRegistration.ipynb).

In [ ]:
import threading
import time

import dask
import itk
import matplotlib.pyplot as plt
import numpy as np
import psutil
from ngff_zarr import (
    NgffImage,
    from_ngff_zarr,
    itk_transform_resample,
    ngff_image_to_itk_image,
    to_multiscales,
    to_ngff_zarr,
)

%matplotlib inline

## Open the pyramids over S3

The bucket is public, so `storage_options={"anon": True}` reads it without AWS
credentials. Nothing is fetched here beyond the metadata.

Each block in flight holds its own moving-image crop, the moving chunks it
shares with its neighbours, and an itkwasm instance, roughly 0.5 GB per 256³
block. The number of concurrent blocks, not the image size, therefore sets the
peak memory: eight keeps an S3 link busy and stays under 10 GB at full
resolution.

In [ ]:
AIND = "s3://aind-open-data"
FIXED_URI = (
    f"{AIND}/exaSPIM_773889_2026-04-10_15-04-57_processed_2026-07-08_23-41-18"
    "/fusion2halves/SPIM.ome.zarr"
)
MOVING_URI = (
    f"{AIND}/exaSPIM_822174_2026-04-28_12-29-55_processed_2026-07-09_03-49-09"
    "/fusion2halves/SPIM.ome.zarr"
)

dask.config.set(scheduler="threads", num_workers=8)

storage_options = {"anon": True}
fixed_multiscales = from_ngff_zarr(FIXED_URI, storage_options=storage_options)
moving_multiscales = from_ngff_zarr(MOVING_URI, storage_options=storage_options)


def spatial(image: NgffImage) -> NgffImage:
    "Drop the singleton t and c axes: registration works on the 3D z, y, x volume."
    keep = [dim for dim in image.dims if dim in ("z", "y", "x")]
    index = tuple(slice(None) if dim in keep else 0 for dim in image.dims)
    return NgffImage(
        data=image.data[index],
        dims=keep,
        scale={dim: image.scale[dim] for dim in keep},
        translation={dim: image.translation[dim] for dim in keep},
        name=image.name,
        axes_units={dim: image.axes_units[dim] for dim in keep}
        if image.axes_units
        else None,
    )


fixed_levels = [spatial(image) for image in fixed_multiscales.images]
moving_levels = [spatial(image) for image in moving_multiscales.images]

print(
    f"{'level':>5} {'shape (z, y, x)':>20} {'voxels':>10} {'chunks':>18}  spacing (µm)"
)
for level, image in enumerate(fixed_levels):
    print(
        f"{level:>5} {image.data.shape!s:>20} {image.data.size / 1e6:>8.1f} M"
        f" {image.data.chunksize!s:>18}  {image.scale}"
    )

## Register at the coarsest level

Level 3 of each brain is 64 × 166 × 222 voxels at 320 × 240 × 240 µm.
`ngff_image_to_itk_image` fetches it from S3 and hands Elastix a native
`itk.Image`; a rigid + affine registration takes a couple of seconds. The
transform lives in physical space (micrometers), so it applies unchanged to
the finer levels, whose grids differ from level 3 in spacing *and* origin
(these stores use a centre-of-voxel convention, so each level's translation
shifts by half the spacing change).

In [ ]:
COARSE = 3

fixed_coarse = ngff_image_to_itk_image(fixed_levels[COARSE], wasm=False).astype(itk.F)
moving_coarse = ngff_image_to_itk_image(moving_levels[COARSE], wasm=False).astype(itk.F)

parameter_object = itk.ParameterObject.New()
parameter_object.AddParameterMap(itk.ParameterObject.GetDefaultParameterMap("rigid"))
parameter_object.AddParameterMap(itk.ParameterObject.GetDefaultParameterMap("affine"))

start = time.perf_counter()
registration = itk.ElastixRegistrationMethod[
    type(fixed_coarse), type(moving_coarse)
].New(
    fixed_image=fixed_coarse,
    moving_image=moving_coarse,
    parameter_object=parameter_object,
)
registration.SetLogToConsole(False)
registration.Update()
print(f"Registration at level {COARSE}: {time.perf_counter() - start:.1f} s")

# ConvertToItkTransform hands back the itk.Transform base; the cast exposes the
# composite interface. The transform maps fixed points into moving space.
brain_transform = itk.CompositeTransform[itk.D, 3].cast(
    registration.ConvertToItkTransform(registration.GetCombinationTransform())
)

In [ ]:
fixed_coarse_array = itk.array_from_image(fixed_coarse)
moving_on_fixed = itk.array_from_image(
    itk.resample_image_filter(
        moving_coarse, use_reference_image=True, reference_image=fixed_coarse
    )
)
registered_coarse = itk.array_from_image(registration.GetOutput())

fig, axs = plt.subplots(2, 3, figsize=[15, 8])
for column, (title, volume) in enumerate(
    [
        ("Fixed", fixed_coarse_array),
        ("Moving (same grid)", moving_on_fixed),
        ("Registered", registered_coarse),
    ]
):
    vmax = np.percentile(volume, 99.5)
    axs[0, column].imshow(volume[volume.shape[0] // 2], cmap="gray", vmax=vmax)
    axs[0, column].set_title(f"{title}: axial")
    axs[1, column].imshow(
        volume[:, volume.shape[1] // 2, :], cmap="gray", vmax=vmax, aspect=320 / 240.64
    )
    axs[1, column].set_title(f"{title}: coronal")
plt.tight_layout()
plt.show()

## Resample a fine level, streaming from S3 to a local OME-Zarr

`itk_transform_resample` returns a lazy `NgffImage` on the fixed grid; nothing
has been read yet. `to_ngff_zarr` then drives the computation: each 256³
output block reads the S3 chunks inside its own moving-image bounding box,
resamples them, and is written to disk as soon as it is done, with eight
blocks in flight at a time.

`FINE` selects the level. Level 1 (150 million voxels, 12 blocks) keeps the
cell to a couple of minutes on a typical connection; level 0 (1.2 billion
voxels, 126 blocks) runs through the same code and is what the benchmark
below reports.

In [ ]:
FINE = 1

resampled_brain = itk_transform_resample(
    brain_transform, fixed_levels[FINE], moving_levels[FINE]
)
print(
    f"Output grid: shape={resampled_brain.data.shape}, blocks={resampled_brain.data.npartitions}"
)

output_store = f"registered_822174_on_773889_level{FINE}.ome.zarr"
start = time.perf_counter()
to_ngff_zarr(
    output_store, to_multiscales(resampled_brain, scale_factors=[]), overwrite=True
)
elapsed = time.perf_counter() - start
print(
    f"Wrote {output_store} in {elapsed:.0f} s "
    f"({resampled_brain.data.size / elapsed / 1e6:.2f} Mvoxel/s end to end)"
)

In [ ]:
# Read the result back from disk and compare with the fixed brain on the same grid.
registered_brain = from_ngff_zarr(output_store).images[0]
z = registered_brain.data.shape[0] // 2
registered_slice = np.asarray(registered_brain.data[z])
fixed_slice = np.asarray(fixed_levels[FINE].data[z])

# Checkerboard: alternating 128-pixel tiles of the fixed and registered slices.
tiles = (np.indices(fixed_slice.shape) // 128).sum(axis=0) % 2 == 0
checkerboard = np.where(tiles, fixed_slice, registered_slice)

fig, axs = plt.subplots(1, 3, figsize=[18, 5])
vmax = np.percentile(fixed_slice, 99.5)
axs[0].imshow(fixed_slice, cmap="gray", vmax=vmax)
axs[0].set_title(f"Fixed, level {FINE}, z={z}")
axs[1].imshow(registered_slice, cmap="gray", vmax=vmax)
axs[1].set_title("Moving resampled onto the fixed grid")
axs[2].imshow(checkerboard, cmap="gray", vmax=vmax)
axs[2].set_title("Checkerboard")
plt.tight_layout()
plt.show()

## Benchmark

The same resample, timed level by level, with the peak resident memory of the
process sampled while it runs. Every level goes through the identical code
path; only the grid changes. `BENCHMARK_LEVELS` defaults to the two coarse
levels so the cell stays quick; extend it to `[3, 2, 1, 0]` to reproduce the
table below.

In [ ]:
class PeakMemory:
    "Peak resident set size of this process while the block runs, sampled every 50 ms."

    def __enter__(self):
        self.process = psutil.Process()
        self.peak = self.process.memory_info().rss
        self._stop = False

        def sample():
            while not self._stop:
                self.peak = max(self.peak, self.process.memory_info().rss)
                time.sleep(0.05)

        self._thread = threading.Thread(target=sample, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop = True
        self._thread.join()


def benchmark(level):
    image = itk_transform_resample(
        brain_transform, fixed_levels[level], moving_levels[level]
    )
    store = f"benchmark_level{level}.ome.zarr"
    with PeakMemory() as memory:
        start = time.perf_counter()
        to_ngff_zarr(store, to_multiscales(image, scale_factors=[]), overwrite=True)
        elapsed = time.perf_counter() - start
    return {
        "level": level,
        "shape": image.data.shape,
        "voxels (M)": round(image.data.size / 1e6, 1),
        "blocks": image.data.npartitions,
        "seconds": round(elapsed, 1),
        "Mvoxel/s": round(image.data.size / elapsed / 1e6, 2),
        "peak RSS (GB)": round(memory.peak / 1e9, 2),
    }


BENCHMARK_LEVELS = [3, 2]

results = [benchmark(level) for level in BENCHMARK_LEVELS]
for row in results:
    print(row)

Measured with `BENCHMARK_LEVELS = [3, 2, 1, 0]` on a 24-core laptop, eight
blocks in flight, once reading the stores from local disk (`aws s3 sync
--no-sign-request` of the same two prefixes) and once streaming them from
`us-west-2` over a link in Europe that sustains 1 to 3 MB/s (the S3 wall-clock
moves with the link; the level 1 and 0 rows come from a scripted run of the same
cells):

| level | shape (z, y, x) | voxels | blocks | local disk | streamed from S3 |
|---:|:---|---:|---:|:---|:---|
| 3 | 64 × 166 × 222 | 2.4 M | 1 | 0.3 s, peak 1.2 GB | 4 s, peak 1.8 GB |
| 2 | 128 × 332 × 444 | 19 M | 4 | 0.6 s (31 Mvoxel/s), 2.0 GB | 13 s (1.4 Mvoxel/s), 2.0 GB |
| 1 | 256 × 665 × 888 | 151 M | 12 | 2.6 s (57 Mvoxel/s), 4.0 GB | 98 s (1.5 Mvoxel/s), 2.9 GB |
| 0 | 513 × 1331 × 1776 | 1213 M | 126 | 14 s (87 Mvoxel/s), 9.6 GB | 710 s (1.7 Mvoxel/s), 6.8 GB |

What the table says:

- **Memory follows concurrency, not image size.** The output grows from 5 MB
  to 2.4 GB and the moving image from 5 MB to 2.4 GB, yet the peak stays
  under 10 GB with eight blocks in flight. Halve `num_workers` and the peak
  roughly halves. Loading the level 0 moving image in-core and calling
  `itk.resample_image_filter` once needs the whole 2.4 GB input plus a 2.4 GB
  output before any float internals.
- **Every moving chunk crosses the network once.** With 256³ chunks on both
  sides and an affine transform, each moving chunk lies in the region of 4 to
  6 output blocks; because the blocks share the chunks in one Dask graph, the
  level 0 run issued exactly 126 chunk reads for 126 chunks. Over S3 the
  wall-clock is then the download of the moving level (1.5 GB compressed at
  level 0), and the same code runs at disk speed next to the bucket or on a
  local copy.
- **The resample itself is fast.** From local disk level 0 runs at 87
  Mvoxel/s including zstd decoding and writing; the same block-wise resample
  with the moving level already in memory reaches 110 to 130 Mvoxel/s, and a
  single native `itk.resample_image_filter` call on in-memory data 590 to 680
  Mvoxel/s. The `itkwasm` cost buys a resample that needs no native ITK build
  and gives identical results across platforms.